# 05. Fine-Tuning Preparation

Prepare data and analysis for future fine-tuning of embedding or LLM models.


In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import json
from src.rag_pipeline import NutritionRAG
from config.settings import RAGConfig

/Users/soham/Desktop/nutribot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Collect Training Pairs


In [2]:
training_pairs = []
test_queries = [
    {'query': 'How much protein do I need daily?', 'relevant_doc_ids': [0]},
    {'query': 'What are good sources of healthy fats?', 'relevant_doc_ids': [2]},
    {'query': 'How much water should I drink?', 'relevant_doc_ids': [4]},
]

df = pd.read_csv('../data/processed/nutrition_docs.csv')
documents = df['content'].tolist()

for test in test_queries:
    query = test['query']
    relevant_ids = test['relevant_doc_ids']
    for doc_id, doc in enumerate(documents):
        is_relevant = doc_id in relevant_ids
        training_pairs.append({'query': query, 'document': doc, 'relevant': is_relevant})

pairs_df = pd.DataFrame(training_pairs)
print(f"Created {len(pairs_df)} query-document pairs")
print(f"Positive pairs: {pairs_df['relevant'].sum()}")
print(f"Negative pairs: {(~pairs_df['relevant']).sum()}")

Created 30 query-document pairs
Positive pairs: 3
Negative pairs: 27


## Analyze Query Difficulty


In [3]:
config = RAGConfig()
rag = NutritionRAG(config)
rag.setup_hybrid_retriever(documents)

query_stats = []
for test in test_queries:
    query = test['query']
    result = rag.query(query, top_k=10)
    stats = {
        'query': query,
        'query_length': len(query.split()),
        'num_retrieved': len(result['retrieved_docs']),
        'top_score': max(result['scores']) if result['scores'] else 0,
        'avg_score': np.mean(result['scores']) if result['scores'] else 0,
        'score_variance': np.var(result['scores']) if len(result['scores']) > 1 else 0,
    }
    query_stats.append(stats)

stats_df = pd.DataFrame(query_stats)
print("Query Statistics for Fine-Tuning:")
print(stats_df.to_string(index=False))

2026-06-15 20:26:59,856 - src.rag_pipeline - INFO - Initializing NutritionRAG pipeline
2026-06-15 20:26:59,858 - src.embeddings.factory - INFO - Creating minilm embedder
2026-06-15 20:26:59,859 - src.embeddings.minilm - INFO - Loading MiniLM embedder from sentence-transformers/all-MiniLM-L6-v2
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19750.99it/s]
2026-06-15 20:27:02,528 - src.embeddings.minilm - INFO - MiniLM embedder loaded successfully on cpu
2026-06-15 20:27:02,529 - src.rag_pipeline - INFO - Embedder initialized: sentence-transformers/all-MiniLM-L6-v2 (384d)
2026-06-15 20:27:02,530 - src.rag_pipeline - INFO - Using hybrid retriever (BM25 + semantic)
2026-06-15 20:27:02,731 - src.rag_pipeline - ERROR - Failed to initialize Groq LLM: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable
2026-06-15 20:27:02,732 - src.rag_pipeline - INFO - RAG pipeline initialized successfully
2026-06-15 20:27:0

Query Statistics for Fine-Tuning:
                                 query  query_length  num_retrieved  top_score  avg_score  score_variance
     How much protein do I need daily?             7              4   0.843340   0.456755        0.050002
What are good sources of healthy fats?             7              4   0.770256   0.465115        0.031855
        How much water should I drink?             6              3   0.794875   0.491406        0.046051


## Export Fine-Tuning Dataset


In [4]:
finetune_data = []
for pair in training_pairs:
    finetune_data.append({
        'query': pair['query'],
        'document': pair['document'],
        'label': 1 if pair['relevant'] else 0,
    })

os.makedirs('../data/finetune', exist_ok=True)

with open('../data/finetune/embedding_pairs.jsonl', 'w') as f:
    for item in finetune_data:
        f.write(json.dumps(item) + '\\n')

print(f"Saved {len(finetune_data)} fine-tuning pairs to data/finetune/embedding_pairs.jsonl")

pd.DataFrame(finetune_data).to_csv('../data/finetune/embedding_pairs.csv', index=False)
print("Also saved as CSV format")

Saved 30 fine-tuning pairs to data/finetune/embedding_pairs.jsonl
Also saved as CSV format


## Hard Negative Mining


In [5]:
hard_negatives = []

for test in test_queries:
    query = test['query']
    relevant_ids = test['relevant_doc_ids']
    result = rag.query(query, top_k=5)
    for i, (doc, score) in enumerate(zip(result['retrieved_docs'], result['scores'])):
        doc_id = documents.index(doc)
        if doc_id not in relevant_ids and score > 0.3:
            hard_negatives.append({
                'query': query,
                'document': doc,
                'score': score,
                'rank': i + 1,
            })

if hard_negatives:
    hn_df = pd.DataFrame(hard_negatives)
    print(f"Found {len(hn_df)} hard negatives (to include in fine-tuning):")
    print(hn_df.to_string())
else:
    print("No hard negatives found (retriever is working well!)")

2026-06-15 20:27:07,360 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:27:07,360 - src.retrieval.hybrid - INFO - Retrieved 4 relevant documents
2026-06-15 20:27:07,361 - src.rag_pipeline - ERROR - LLM not initialized
2026-06-15 20:27:07,362 - src.rag_pipeline - INFO - Query completed in 0.02s (retrieval: 0.01s, LLM: 0.00s)
2026-06-15 20:27:07,370 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:27:07,370 - src.retrieval.hybrid - INFO - Retrieved 4 relevant documents
2026-06-15 20:27:07,371 - src.rag_pipeline - ERROR - LLM not initialized
2026-06-15 20:27:07,371 - src.rag_pipeline - INFO - Query completed in 0.01s (retrieval: 0.01s, LLM: 0.00s)
2026-06-15 20:27:07,382 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:27:07,383 - src.retrieval.hybrid - INFO - Retrieved 3 relevant documents
2026-06-15 20:27:07,384 - src.rag_pipeline - ERROR - LLM not initialized
2026-06-15 20:27:07,384 - sr

Found 8 hard negatives (to include in fine-tuning):
                                    query                                                                                                                                                                                              document     score  rank
0       How much protein do I need daily?                      Fiber is important for digestive health. Aim for 25-30g daily from sources like whole grains, legumes, vegetables, and fruits. Fiber helps with satiety and blood sugar control.  0.344457     2
1       How much protein do I need daily?  Water intake recommendations vary, but the general guideline is 8 glasses per day. However, needs depend on activity level, climate, and individual factors. Drink enough to keep urine pale yellow.  0.332464     3
2       How much protein do I need daily?            Sugar consumption should be minimized. Added sugars provide empty calories and can lead to weight gain and increased disease ri

## Fine-Tuning Recommendations


In [6]:
recommendations = f"""
Fine-Tuning Strategy Recommendations:

1. EMBEDDING MODEL FINE-TUNING (BGE):
   - Use query-document pairs as training data
   - Include hard negatives for contrast learning
   - Framework: Sentence Transformers with MultipleNegativesRankingLoss
   - Expected improvement: 5-15% on domain queries

2. LLM FINE-TUNING:
   - Collect Q&A pairs with highly-rated responses
   - Methods: LoRA, QLoRA, or full fine-tuning

3. DATA REQUIREMENTS:
   - Current: {len(finetune_data)} pairs available
   - Current: {len(hard_negatives)} hard negatives

4. EVALUATION SETUP:
   - Hold out 20% for testing
   - Measure MRR, NDCG@5, Precision@5
   - Compare before/after fine-tuning
"""

print(recommendations)


Fine-Tuning Strategy Recommendations:

1. EMBEDDING MODEL FINE-TUNING (BGE):
   - Use query-document pairs as training data
   - Include hard negatives for contrast learning
   - Framework: Sentence Transformers with MultipleNegativesRankingLoss
   - Expected improvement: 5-15% on domain queries

2. LLM FINE-TUNING:
   - Collect Q&A pairs with highly-rated responses
   - Methods: LoRA, QLoRA, or full fine-tuning

3. DATA REQUIREMENTS:
   - Current: 30 pairs available
   - Current: 8 hard negatives

4. EVALUATION SETUP:
   - Hold out 20% for testing
   - Measure MRR, NDCG@5, Precision@5
   - Compare before/after fine-tuning

